In [0]:
%pip install timezonefinder

In [ ]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from timezonefinder import TimezoneFinder
import math

CATALOG = "workspace"
SCHEMA = "silver"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

print(f"Gold notebook initialized: {CATALOG}.{SCHEMA}")

In [0]:
silver_tables = [
    table.name
    for table in spark.catalog.listTables(f"{CATALOG}.{SCHEMA}")
]

for table_name in sorted(silver_tables):
    print(f"\n{'=' * 60}")
    print(f"TABLE: {CATALOG}.{SCHEMA}.{table_name}")
    print("=" * 60)

    spark.table(f"{CATALOG}.{SCHEMA}.{table_name}").printSchema()

if not silver_tables:
    print(f"No tables found in {CATALOG}.{SCHEMA}.")

In [0]:
flights_df = spark.table("workspace.silver.flights")
airports_df = spark.table("workspace.silver.airports")
weather_df = spark.table("workspace.silver.weather")

In [0]:
weather_columns = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "wind_speed_10m",
    "wind_gusts_10m",
    "visibility",
    "cloud_cover",
]

# Duplicate weather keys would multiply flight rows during the join.
duplicate_weather_keys = (
    weather_df
    .groupBy("airport_code", "weather_timestamp")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_weather_keys.limit(1).count():
    raise ValueError(
        "Duplicate airport/hour weather records found. Resolve before joining."
    )

flights_with_hour_df = flights_df.withColumn(
    "scheduled_departure_local_hour",
    F.date_trunc("hour", F.col("scheduled_departure_local_ts"))
    .cast("timestamp_ntz"),
)

origin_weather_df = weather_df.select(
    F.col("airport_code").alias("origin_weather_airport"),
    F.col("weather_timestamp").alias("origin_weather_timestamp"),
    *[
        F.col(column).alias(f"origin_weather_{column}")
        for column in weather_columns
    ],
)

gold_df = (
    flights_with_hour_df
    .join(
        origin_weather_df,
        (
            flights_with_hour_df["origin"]
            == origin_weather_df["origin_weather_airport"]
        )
        & (
            flights_with_hour_df["scheduled_departure_local_hour"]
            == origin_weather_df["origin_weather_timestamp"]
        ),
        "left",
    )
    .withColumn(
        "origin_weather_matched",
        F.col("origin_weather_airport").isNotNull(),
    )
    .drop("origin_weather_airport")
)

display(
    gold_df.groupBy("origin_weather_matched").count()
)

In [0]:
display(
    spark.table("workspace.bronze.weather")
    .filter(F.col("airport_code").isin("PHF", "VRB"))
    .groupBy("airport_code")
    .count()
)

In [0]:
flight_count = flights_df.count()
gold_count = gold_df.count()

if gold_count != flight_count:
    raise ValueError(
        f"Row count changed during weather join: "
        f"{flight_count:,} flights → {gold_count:,} Gold rows"
    )

print(f"Validated: {gold_count:,} flight rows preserved.")

display(
    gold_df
    .groupBy("origin_weather_matched")
    .count()
)

In [0]:
display(
    gold_df
    .groupBy("reporting_airline")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
carrier_mapping = [
    # Code, airline name, network type, business model
    ("WN", "Southwest", "point_to_point", "LCC"),
    ("DL", "Delta", "hub_and_spoke", "legacy"),
    ("AA", "American", "hub_and_spoke", "legacy"),
    ("UA", "United", "hub_and_spoke", "legacy"),
    ("OO", "SkyWest", "hub_and_spoke", "regional"),
    ("YX", "Republic", "hub_and_spoke", "regional"),
    ("MQ", "Envoy", "hub_and_spoke", "regional"),
    ("B6", "JetBlue", "point_to_point", "hybrid"),
    ("AS", "Alaska", "hub_and_spoke", "legacy"),
    ("NK", "Spirit", "point_to_point", "ULCC"),
    ("OH", "PSA", "hub_and_spoke", "regional"),
    ("F9", "Frontier", "point_to_point", "ULCC"),
    ("9E", "Endeavor", "hub_and_spoke", "regional"),
    ("G4", "Allegiant", "point_to_point", "ULCC"),
    ("HA", "Hawaiian", "hub_and_spoke", "legacy"),
]

carrier_features_df = spark.createDataFrame(
    carrier_mapping,
    schema="""
        reporting_airline string,
        carrier_name string,
        carrier_network_type string,
        carrier_business_model string
    """,
)

# Remove previously created features so this cell can be rerun.
gold_df = (
    gold_df
    .drop(
        "carrier_name",
        "carrier_network_type",
        "carrier_business_model",
    )
    .join(
        F.broadcast(carrier_features_df),
        on="reporting_airline",
        how="left",
    )
)

unmapped_carriers = (
    gold_df
    .filter(F.col("carrier_business_model").isNull())
    .select("reporting_airline")
    .distinct()
)

if unmapped_carriers.limit(1).count():
    display(unmapped_carriers)
    raise ValueError("Unmapped carriers found. Update carrier_mapping.")

display(
    gold_df
    .groupBy(
        "reporting_airline",
        "carrier_name",
        "carrier_network_type",
        "carrier_business_model",
    )
    .count()
    .orderBy(F.desc("count"))
)



departure_angle = (
    2 * math.pi
    * (
        F.col("scheduled_departure_hour") * 60
        + F.col("scheduled_departure_minute")
    )
    / 1440
)

month_angle = (
    2 * math.pi
    * (F.month("scheduled_departure_local_date") - 1)
    / 12
)

gold_df = (
    gold_df
    .withColumn(
        "route",
        F.when(
            F.col("origin").isNotNull() & F.col("dest").isNotNull(),
            F.concat(F.col("origin"), F.lit("_"), F.col("dest")),
        ),
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("scheduled_departure_local_date").isin(1, 7),
    )
    .withColumn("departure_time_sin", F.sin(departure_angle))
    .withColumn("departure_time_cos", F.cos(departure_angle))
    .withColumn("month_sin", F.sin(month_angle))
    .withColumn("month_cos", F.cos(month_angle))
)

display(
    gold_df.select(
        "flight_date",
        "reporting_airline",
        "route",
        "scheduled_departure_hour",
        "scheduled_departure_minute",
        "is_weekend",
        "departure_time_sin",
        "departure_time_cos",
        "month_sin",
        "month_cos",
    ).limit(20)
)

arrival_delay_eligible = (
    (F.col("is_cancelled") == F.lit(False))
    & (F.col("is_diverted") == F.lit(False))
    & F.col("arrdel15").isin(0.0, 1.0)
    & (F.col("dq_delay_inconsistent") == F.lit(False))
)

gold_df = (
    gold_df
    .withColumn(
        "arrival_delay_model_eligible",
        F.coalesce(arrival_delay_eligible, F.lit(False)),
    )
    .withColumn(
        "arrival_delay_label",
        F.when(
            F.col("arrival_delay_model_eligible"),
            F.col("arrdel15").cast("int"),
        ),
    )
)

display(
    gold_df
    .groupBy(
        "arrival_delay_model_eligible",
        "arrival_delay_label",
    )
    .count()
)

In [0]:
origin_hour_window = Window.partitionBy(
    "origin",
    "scheduled_departure_local_hour",
)

origin_day_window = Window.partitionBy(
    "origin",
    "scheduled_departure_local_date",
)

carrier_origin_day_window = Window.partitionBy(
    "reporting_airline",
    "origin",
    "scheduled_departure_local_date",
)

valid_origin_hour = (
    F.col("origin").isNotNull()
    & F.col("scheduled_departure_local_hour").isNotNull()
)

valid_origin_day = (
    F.col("origin").isNotNull()
    & F.col("scheduled_departure_local_date").isNotNull()
)

gold_df = (
    gold_df
    .withColumn(
        "origin_scheduled_departures_hour",
        F.when(
            valid_origin_hour,
            F.count("*").over(origin_hour_window),
        ),
    )
    .withColumn(
        "origin_scheduled_departures_day",
        F.when(
            valid_origin_day,
            F.count("*").over(origin_day_window),
        ),
    )
    .withColumn(
        "carrier_origin_scheduled_departures_day",
        F.when(
            valid_origin_day
            & F.col("reporting_airline").isNotNull(),
            F.count("*").over(carrier_origin_day_window),
        ),
    )
    .withColumn(
        "carrier_origin_departure_share_day",
        F.col("carrier_origin_scheduled_departures_day")
        / F.col("origin_scheduled_departures_day"),
    )
)

display(
    gold_df.select(
        "origin",
        "reporting_airline",
        "scheduled_departure_local_date",
        "scheduled_departure_local_hour",
        "origin_scheduled_departures_hour",
        "origin_scheduled_departures_day",
        "carrier_origin_scheduled_departures_day",
        "carrier_origin_departure_share_day",
    ).limit(20)
)

In [0]:
gold_df = (
    gold_df
    .withColumn(
        "origin_has_rain",
        (F.col("origin_weather_rain") > 0).cast("int"),
    )
    .withColumn(
        "origin_has_snow",
        (F.col("origin_weather_snowfall") > 0).cast("int"),
    )
    .withColumn(
        "origin_wind_precipitation_interaction",
        F.col("origin_weather_wind_speed_10m")
        * F.col("origin_weather_precipitation"),
    )
    .withColumn(
        "origin_gust_speed_difference",
        F.col("origin_weather_wind_gusts_10m")
        - F.col("origin_weather_wind_speed_10m"),
    )
)

display(
    gold_df.select(
        "origin",
        "scheduled_departure_local_hour",
        "origin_weather_matched",
        "origin_has_rain",
        "origin_has_snow",
        "origin_wind_precipitation_interaction",
        "origin_gust_speed_difference",
    ).limit(20)
)

In [0]:
airports_df.printSchema()

In [0]:
 

timezone_finder = TimezoneFinder()

# Airport data is small enough to process on the driver.
airport_coordinates = (
    airports_df
    .select("airport_code", "latitude", "longitude")
    .distinct()
    .collect()
)

airport_timezone_rows = []

for row in airport_coordinates:
    latitude = row["latitude"]
    longitude = row["longitude"]

    timezone = None

    if (
        latitude is not None
        and longitude is not None
        and -90 <= latitude <= 90
        and -180 <= longitude <= 180
    ):
        timezone = timezone_finder.timezone_at(
            lat=latitude,
            lng=longitude,
        )

    airport_timezone_rows.append(
        (row["airport_code"], timezone)
    )

airport_timezones_df = spark.createDataFrame(
    airport_timezone_rows,
    schema="airport_code string, airport_timezone string",
)

# Ensure the lookup cannot multiply flight rows during joins.
if (
    airport_timezones_df
    .groupBy("airport_code")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise ValueError("Duplicate airport codes in time-zone lookup.")

display(
    airport_timezones_df.orderBy("airport_code")
)

display(
    airport_timezones_df.filter(
        F.col("airport_timezone").isNull()
    )
)

In [0]:
origin_timezones_df = airport_timezones_df.select(
    F.col("airport_code").alias("origin"),
    F.col("airport_timezone").alias("origin_timezone"),
)

destination_timezones_df = airport_timezones_df.select(
    F.col("airport_code").alias("dest"),
    F.col("airport_timezone").alias("destination_timezone"),
)

gold_df = (
    gold_df
    .drop("origin_timezone", "destination_timezone")
    .join(
        F.broadcast(origin_timezones_df),
        on="origin",
        how="left",
    )
    .join(
        F.broadcast(destination_timezones_df),
        on="dest",
        how="left",
    )
)

# Check for flight airports missing a time zone.
display(
    gold_df
    .filter(
        F.col("origin_timezone").isNull()
        | F.col("destination_timezone").isNull()
    )
    .select(
        "origin",
        "origin_timezone",
        "dest",
        "destination_timezone",
    )
    .distinct()
)

display(
    gold_df.select(
        "origin",
        "origin_timezone",
        "dest",
        "destination_timezone",
        "scheduled_departure_local_ts",
        "crselapsedtime",
    ).limit(20)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

gold_df = (
    gold_df
    .withColumn(
        "scheduled_departure_utc_ts",
        F.expr("""
            convert_timezone(
                origin_timezone,
                'UTC',
                scheduled_departure_local_ts
            )
        """),
    )
    .withColumn(
        "_scheduled_arrival_utc_candidate",
        F.when(
            F.col("scheduled_departure_utc_ts").isNotNull()
            & F.col("crselapsedtime").isNotNull()
            & (F.col("crselapsedtime") > 0)
            & (F.col("crselapsedtime") == F.floor("crselapsedtime")),
            F.expr("""
                timestampadd(
                    MINUTE,
                    CAST(crselapsedtime AS INT),
                    scheduled_departure_utc_ts
                )
            """),
        ),
    )
    .withColumn(
        "_scheduled_arrival_local_candidate",
        F.expr("""
            convert_timezone(
                'UTC',
                destination_timezone,
                _scheduled_arrival_utc_candidate
            )
        """),
    )
    .withColumn(
        "scheduled_arrival_schedule_valid",
        F.coalesce(
            F.col("_scheduled_arrival_local_candidate").isNotNull()
            & F.col("crsarrtime_valid")
            & (
                F.hour("_scheduled_arrival_local_candidate")
                == F.col("scheduled_arrival_hour")
            )
            & (
                F.minute("_scheduled_arrival_local_candidate")
                == F.col("scheduled_arrival_minute")
            ),
            F.lit(False),
        ),
    )
    .withColumn(
        "dq_scheduled_arrival_mismatch",
        F.coalesce(
            F.col("_scheduled_arrival_local_candidate").isNotNull()
            & F.col("crsarrtime_valid")
            & ~F.col("scheduled_arrival_schedule_valid"),
            F.lit(False),
        ),
    )
    .withColumn(
        "scheduled_arrival_utc_ts",
        F.when(
            F.col("scheduled_arrival_schedule_valid"),
            F.col("_scheduled_arrival_utc_candidate"),
        ),
    )
    .withColumn(
        "scheduled_arrival_local_ts",
        F.when(
            F.col("scheduled_arrival_schedule_valid"),
            F.col("_scheduled_arrival_local_candidate"),
        ),
    )
    .drop(
        "_scheduled_arrival_utc_candidate",
        "_scheduled_arrival_local_candidate",
    )
    .withColumn(
        "scheduled_arrival_local_date",
        F.to_date("scheduled_arrival_local_ts"),
    )
    .withColumn(
        "scheduled_arrival_local_hour",
        F.date_trunc("hour", "scheduled_arrival_local_ts")
        .cast("timestamp_ntz"),
    )
    .withColumn(
        "scheduled_arrival_utc_hour",
        F.date_trunc("hour", "scheduled_arrival_utc_ts")
        .cast("timestamp_ntz"),
    )
)


display(
    gold_df.groupBy(
        "scheduled_arrival_schedule_valid",
        "dq_scheduled_arrival_mismatch",
    ).count()
)


destination_hour_window = Window.partitionBy(
    "dest",
    "scheduled_arrival_utc_hour",
)

gold_df = gold_df.withColumn(
    "destination_scheduled_arrivals_hour",
    F.when(
        F.col("dest").isNotNull()
        & F.col("scheduled_arrival_utc_hour").isNotNull(),
        F.count("*").over(destination_hour_window),
    ),
)

weather_columns = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "wind_speed_10m",
    "wind_gusts_10m",
    "visibility",
    "cloud_cover",
]

duplicate_weather_keys = (
    weather_df
    .groupBy("airport_code", "weather_timestamp")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_weather_keys.limit(1).count():
    raise ValueError(
        "Duplicate airport/hour weather records found."
    )

destination_weather_df = weather_df.select(
    F.col("airport_code").alias("destination_weather_airport"),
    F.col("weather_timestamp")
    .cast("timestamp_ntz")
    .alias("destination_weather_timestamp"),
    *[
        F.col(column).alias(f"destination_weather_{column}")
        for column in weather_columns
    ],
)

gold_df = gold_df.drop(
    "destination_weather_airport",
    "destination_weather_timestamp",
    "destination_weather_matched",
    *[
        f"destination_weather_{column}"
        for column in weather_columns
    ],
)


gold_df = (
    gold_df
    .join(
        destination_weather_df,
        (
            gold_df["dest"]
            == destination_weather_df["destination_weather_airport"]
        )
        & (
            gold_df["scheduled_arrival_local_hour"]
            == destination_weather_df["destination_weather_timestamp"]
        ),
        "left",
    )
    .withColumn(
        "destination_weather_matched",
        F.col("destination_weather_airport").isNotNull(),
    )
    .drop("destination_weather_airport")
)

display(
    gold_df.select(
        "origin",
        "dest",
        "scheduled_departure_local_ts",
        "scheduled_arrival_local_ts",
        "destination_scheduled_arrivals_hour",
        "destination_weather_matched",
        "destination_weather_temperature_2m",
        "destination_weather_precipitation",
        "destination_weather_wind_speed_10m",
    ).limit(20)
)

display(
    gold_df
    .groupBy("destination_weather_matched")
    .count()
)

In [0]:
from pyspark.sql import functions as F


original_count = flights_df.count()
gold_count = gold_df.count()

print(f"Original flights: {original_count:,}")
print(f"Gold rows:        {gold_count:,}")
print(f"Difference:       {gold_count - original_count:,}")

if gold_count != original_count:
    raise ValueError("Gold row count differs from the original flight count.")

display(
    gold_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("record_id").alias("distinct_record_ids"),
        F.sum(
            F.col("record_id").isNull().cast("long")
        ).alias("null_record_ids"),
    )
)


required_features = [
    "origin",
    "dest",
    "reporting_airline",
    "origin_timezone",
    "destination_timezone",
    "carrier_network_type",
    "carrier_business_model",
    "scheduled_departure_local_ts",
    "scheduled_departure_utc_ts",
    "scheduled_arrival_local_ts",
    "scheduled_arrival_utc_ts",
    "origin_scheduled_departures_hour",
    "origin_scheduled_departures_day",
    "carrier_origin_scheduled_departures_day",
    "carrier_origin_departure_share_day",
    "destination_scheduled_arrivals_hour",
]

null_summary = gold_df.agg(
    *[
        F.sum(F.col(column).isNull().cast("long")).alias(column)
        for column in required_features
    ]
)

display(
    null_summary.unpivot(
        ids=[],
        values=required_features,
        variableColumnName="feature",
        valueColumnName="null_count",
    ).orderBy(F.desc("null_count"))
)



invalid_checks = {
    "nonpositive_origin_hour_traffic":
        F.col("origin_scheduled_departures_hour") <= 0,

    "nonpositive_destination_hour_traffic":
        F.col("destination_scheduled_arrivals_hour") <= 0,

    "invalid_carrier_share":
        (F.col("carrier_origin_departure_share_day") <= 0)
        | (F.col("carrier_origin_departure_share_day") > 1),

    "carrier_departures_exceed_airport":
        F.col("carrier_origin_scheduled_departures_day")
        > F.col("origin_scheduled_departures_day"),

    "origin_hour_exceeds_day":
        F.col("origin_scheduled_departures_hour")
        > F.col("origin_scheduled_departures_day"),

    "arrival_not_after_departure":
        F.col("scheduled_arrival_utc_ts")
        <= F.col("scheduled_departure_utc_ts"),

    "eligible_without_valid_label":
        F.col("arrival_delay_model_eligible")
        & (
            F.col("arrival_delay_label").isNull()
            | ~F.col("arrival_delay_label").isin(0, 1)
        ),

    "ineligible_with_label":
        ~F.col("arrival_delay_model_eligible")
        & F.col("arrival_delay_label").isNotNull(),
}

display(
    gold_df.agg(
        *[
            F.sum(
                F.coalesce(condition, F.lit(False)).cast("long")
            ).alias(name)
            for name, condition in invalid_checks.items()
        ]
    )
)

# ------------------------------------------------------------
# 5. Weather join coverage.
# Missing matches are allowed but should be understood.
# ------------------------------------------------------------

display(
    gold_df
    .groupBy(
        "origin_weather_matched",
        "destination_weather_matched",
    )
    .count()
    .orderBy(F.desc("count"))
)

# Matched weather records can still contain missing measurements.
weather_features = [
    f"{location}_weather_{measure}"
    for location in ["origin", "destination"]
    for measure in [
        "temperature_2m",
        "precipitation",
        "rain",
        "snowfall",
        "wind_speed_10m",
        "wind_gusts_10m",
        "visibility",
        "cloud_cover",
    ]
]

display(
    gold_df.agg(
        *[
            F.sum(F.col(column).isNull().cast("long")).alias(column)
            for column in weather_features
        ]
    )
)

# ------------------------------------------------------------
# 6. Label balance and coverage by year.
# ------------------------------------------------------------

display(
    gold_df
    .groupBy("calendar_year")
    .agg(
        F.count("*").alias("total_flights"),
        F.sum(
            F.col("arrival_delay_model_eligible").cast("long")
        ).alias("eligible_flights"),
        F.sum("arrival_delay_label").alias("delayed_flights"),
        F.round(
            100 * F.avg("arrival_delay_label"), 2
        ).alias("eligible_delayed_pct"),
    )
    .orderBy("calendar_year")
)

display(
    gold_df
    .filter(
        F.col("scheduled_arrival_local_date")
        != F.col("scheduled_departure_local_date")
    )
    .select(
        "origin",
        "dest",
        "origin_timezone",
        "destination_timezone",
        "scheduled_departure_local_ts",
        "crsarrtime",
        "crselapsedtime",
        "scheduled_arrival_local_ts",
    )
    .limit(20)
)

In [ ]:
model_df = gold_df.na.drop(how="any", subset=weather_features)

model_row_count = model_df.count()
missing_weather_count = gold_count - model_row_count

print(f"Rows before weather filtering: {gold_count:,}")
print(f"Rows dropped for missing weather: {missing_weather_count:,}")
print(f"Rows retained in model_df: {model_row_count:,}")
